# Cải tiến KAC bằng Spherical: ViT đóng băng và đặc trưng APT — 4 dataset, quy trình Train / Val / Test của nhóm

Code chính thức của APT + head KAC (head lr 1e-3). Sau mỗi task lưu logit KAC và đặc trưng: **ViT đóng băng** (không prompt) và **APT** (CLS sau prompt). Phân tích offline:

| nguồn | tâm lớp | đặc trưng ảnh đánh giá |
|---|---|---|
| `frozen` | ViT không prompt | ViT không prompt |
| `apt_stale` | APT ngay sau task của lớp, không cập nhật | APT của model hiện tại |
| `apt_refresh` | APT của model hiện tại trên mọi ảnh train đã thấy (cần giữ ảnh cũ → chỉ tham chiếu) | APT của model hiện tại |

Mỗi nguồn có `proto` (Spherical thuần), `fuse` (gộp log-xác suất với KAC), `hier` (Spherical chọn task, KAC chọn lớp trong task); thêm `KAC` và `oracle` (KAC biết task id).

**Dữ liệu và quy trình (theo `check-dataset.ipynb` của nhóm):** CIFAR-100 (10×10), CUB-200 (10×20), Stanford Cars (16 + 9×20), ImageNet-R (10×20). Seed s ∈ {0,1,2}: thứ tự lớp = `RandomState(s).permutation`, Val = Fold[s] của 5-fold phân tầng (`base_seed=1993`).
1. **`PHASE = "val"`**: học trên 80% Train, đánh giá trên 20% Val; chọn `k, κ_mult, w, λ, w_in` theo Last trung bình 3 seed → lưu `best_configs_val.json`. Không đụng Test.
2. **`PHASE = "test"`**: học trên 100% Train, đánh giá trên Test, dùng **nguyên** cấu hình đã chọn ở bước 1.

**Kaggle:** Settings → GPU (T4/P100, 1 GPU) + Internet On. Add Input: dataset chứa repo APT (`RepoGit_APT`) và dataset của nhóm cho dataset đang chạy. Kết quả trong `/kaggle/working/apt_spherical/`.
Ước lượng thời gian mỗi seed (T4, fp32 như repo): CUB ~1 giờ, Cars ~2–3 giờ, ImageNet-R ~4 giờ, CIFAR-100 ~8 giờ. Phiên Kaggle tối đa ~12 giờ → nên chạy 1 dataset (CIFAR: 1 seed) mỗi phiên. Phiên sau: Add Input output của phiên trước, ô phân tích sẽ tự gom dump của mọi seed.

## 1. Cấu hình

In [1]:
# ==============================================================================
# 1. CẤU HÌNH
# ==============================================================================
DATASETS = ["CIFAR100"]      # chọn trong ["CIFAR100", "CUB200", "StanfordCars", "ImageNet_R"] (nên 1 dataset mỗi phiên)
PHASE    = "val"           # "val": học 80% Train, đánh giá 20% Val -> chọn cấu hình (KHÔNG đụng Test)
                           # "test": học 100% Train, đánh giá Test với cấu hình đã chọn ở "val" (cần best_configs_val.json)
SEEDS    = [1]       # seed s: thứ tự task = RandomState(s).permutation, Val = Fold[s] (quy trình của nhóm)
HEADS    = ["kac"]
RUN_TRAINING = True       # False: chỉ phân tích dump có sẵn (Add Input output phiên trước) - không cần GPU, vài phút
RUN_SMOKE    = False       # False: bỏ ô chạy thử (~10-20 phút; nên tắt với CIFAR-100)
KAC_GRIDS = 4              # paper KAC: N = 4 RBF
HEAD_LR  = 0.001           # lr riêng của head KAC
APT_DUMP_APT = "2"         # "2": lưu đặc trưng ViT đóng băng + APT (frozen, apt_stale, apt_refresh); "1": bỏ apt_refresh (nhanh hơn)
BATCH, PROMPT_PARAM = 64, 0.01
# Siêu tham số APT lấy từ experiments/*.sh của repo chính thức
HP = {"CIFAR100":     dict(lr=0.004, epochs=30, ema=0.7, config="configs/cifar-100_prompt.yaml"),
      "CUB200":       dict(lr=0.02,  epochs=25, ema=0.7, config="configs/cub200_prompt.yaml"),
      "StanfordCars": dict(lr=0.005, epochs=50, ema=0.7, config="configs/stanfordcars_prompt.yaml"),
      "ImageNet_R":   dict(lr=0.003, epochs=30, ema=0.8, config="configs/imnet-r_prompt.yaml")}
DRIVE_DIR = "/kaggle/working/apt_spherical"
ZIP_PATH  = None           # tự tìm repo APT trong /kaggle/input
assert PHASE in ("val", "test") and set(SEEDS) <= {0, 1, 2, 3, 4} and set(DATASETS) <= set(HP)


## 2. Tìm repo + cài thư viện
Cần GPU và Internet (tải `timm` và trọng số ViT-B/16).

In [2]:
import os, sys, glob, shutil, subprocess, zipfile
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
INPUT_ROOT = "/kaggle/input"
print("Nội dung /kaggle/input:", os.listdir(INPUT_ROOT) if os.path.isdir(INPUT_ROOT) else "KHÔNG CÓ")
if not glob.glob("/tmp/apt_repo/**/run.py", recursive=True):
    runs = glob.glob(f"{INPUT_ROOT}/**/run.py", recursive=True)          # Kaggle thường tự giải nén zip khi upload
    zips = glob.glob(f"{INPUT_ROOT}/**/RepoGit_APT.zip", recursive=True)
    assert runs or zips, "Notebook chưa có dataset chứa repo: bên phải -> Add Input -> chọn dataset đã upload RepoGit_APT.zip"
    os.makedirs("/tmp/apt_repo", exist_ok=True)
    if zips: zipfile.ZipFile(zips[0]).extractall("/tmp/apt_repo")
    else: shutil.copytree(os.path.dirname(runs[0]), "/tmp/apt_repo/Additive-Prompt-Tuning-main")   # /kaggle/input chỉ đọc -> chép ra /tmp để patch
REPO = os.path.dirname(glob.glob("/tmp/apt_repo/**/run.py", recursive=True)[0])
print("Repo:", REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "timm==0.4.12"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyaml", "gdown"], check=True)
import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert torch.cuda.is_available() or not RUN_TRAINING, "Bật GPU: Settings -> Accelerator -> GPU T4 x2 (hoặc P100)"
r = subprocess.run([sys.executable, "-c", """
import sys; sys.path.insert(0,'.')
import timm, torch
from timm.models import vit_base_patch16_224
import timm.models.vision_transformer as v
from models.vit import VisionTransformer
print('timm', timm.__version__); print(v.default_cfgs['vit_base_patch16_224']['url'])
m = VisionTransformer(img_size=224, patch_size=16, embed_dim=768, depth=12, num_heads=12, ckpt_layer=0, drop_path_rate=0)
sd = vit_base_patch16_224(pretrained=True).state_dict(); del sd['head.weight']; del sd['head.bias']
r = m.load_state_dict(sd, strict=False); print('missing', r.missing_keys, 'unexpected', r.unexpected_keys)
assert not r.missing_keys and not r.unexpected_keys
print('OK trọng số khớp')
"""], cwd=REPO, capture_output=True, text=True)
print(r.stdout[-600:], r.stderr[-800:] if r.returncode else "")
assert r.returncode == 0 or not RUN_TRAINING, "Cần bật Internet: Settings -> Internet On"


Nội dung /kaggle/input: ['datasets']
Repo: /tmp/apt_repo/Additive-Prompt-Tuning-main
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 2.1 MB/s eta 0:00:00
torch 2.11.0+cu128 | GPU: Tesla T4
timm 0.4.12
https://storage.googleapis.com/vit_models/augreg/B_16-i21k-300ep-lr_0.001-aug_medium1-wd_0.1-do_0.0-sd_0.0--imagenet2012-steps_20k-lr_0.01-res_224.npz
missing [] unexpected []
OK trọng số khớp
 


## 3. Dữ liệu: chia Train / Val / Test theo quy trình của nhóm
In ra 5 lớp đầu của Task 1 cho từng seed: phải trùng với `check-dataset.ipynb` (ví dụ CUB seed 0: `[18, 170, 107, 98, 177]`).

In [3]:
# ==============================================================================
# DỮ LIỆU: 4 dataset + chia Train/Val/Test theo quy trình chung của nhóm (check-dataset.ipynb)
#   - seed s ∈ {0,1,2}: thứ tự lớp = np.random.RandomState(s).permutation(C), cắt theo TASK_SIZES
#   - Val = Fold[s] của Stratified 5-Fold trong từng lớp (base_seed = 1993), Train = 4 fold còn lại
#   - PHASE="val" : học trên 80% Train, đánh giá trên 20% Val   (chọn cấu hình)
#   - PHASE="test": học trên 100% Train,  đánh giá trên Test     (số liệu báo cáo)
# Ghi ra {WORK}/splits/<ds>/<phase>/seed<s>/{train,test}.json; repo APT đọc qua biến môi trường APT_SPLIT_DIR.
# Nhãn được đánh lại = vị trí của lớp trong thứ tự task -> task k là một khối nhãn liên tiếp (rand_split=False trong repo).
# ==============================================================================
import os, json, glob, pickle, random, collections
import numpy as np
from PIL import Image
from torchvision.datasets import ImageFolder

WORK = DRIVE_DIR
DS_SPEC = {
    "CIFAR100":     dict(C=100, sizes=[10] * 10,       kind="cifar",
                         roots=["/kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python",
                                "/kaggle/input/cifar-100-python/cifar-100-python"]),
    "CUB200":       dict(C=200, sizes=[20] * 10,       kind="folder",
                         roots=["/kaggle/input/datasets/cyizhuo/cub-200-2011-by-classes-folder",
                                "/kaggle/input/cub-200-2011-by-classes-folder"]),
    "StanfordCars": dict(C=196, sizes=[16] + [20] * 9, kind="folder",
                         roots=["/kaggle/input/datasets/jutrera/stanford-car-dataset-by-classes-folder/car_data/car_data",
                                "/kaggle/input/stanford-car-dataset-by-classes-folder/car_data/car_data"]),
    "ImageNet_R":   dict(C=200, sizes=[20] * 10,       kind="folder",
                         roots=["/kaggle/input/datasets/thohintrnhngc/imagenet-r",
                                "/kaggle/input/imagenet-r"]),
}

def find_root(ds):
    sp = DS_SPEC[ds]
    ok = (lambda p: all(os.path.isfile(os.path.join(p, f)) for f in ("train", "test", "meta"))) if sp["kind"] == "cifar" \
         else (lambda p: os.path.isdir(os.path.join(p, "train")) and os.path.isdir(os.path.join(p, "test")))
    for p in sp["roots"]:
        if ok(p): return p
    # dự phòng: tìm trong /kaggle/input theo tên thư mục cuối của đường dẫn chuẩn
    tail = os.path.basename(sp["roots"][0].rstrip("/"))
    for p in glob.glob(f"/kaggle/input/**/{tail}", recursive=True):
        if ok(p): return p
    raise FileNotFoundError(f"{ds}: không thấy dữ liệu. Add Input dataset của nhóm (đường dẫn chuẩn: {sp['roots'][0]})")

def load_source(ds):
    """Trả về (train_items, train_targets, test_items, test_targets, root) theo đúng cách đọc của check-dataset."""
    root = find_root(ds)
    if DS_SPEC[ds]["kind"] == "cifar":
        out = []
        for split in ("train", "test"):
            with open(os.path.join(root, split), "rb") as f:
                entry = pickle.load(f, encoding="latin1")
            y = [int(v) for v in entry["fine_labels"]]
            out += [[f"{split}:{i}" for i in range(len(y))], y]
        return out[0], out[1], out[2], out[3], root
    tr, te = ImageFolder(os.path.join(root, "train")), ImageFolder(os.path.join(root, "test"))
    assert tr.classes == te.classes, f"{ds}: tên lớp train/test không khớp"
    return [p for p, _ in tr.samples], list(tr.targets), [p for p, _ in te.samples], list(te.targets), root

def setup_seed(train_targets, seed, num_classes, task_sizes, n_splits=5, base_seed=1993):
    """Y hệt setup_<dataset>_seed trong check-dataset.ipynb (Phương án A)."""
    assert 0 <= seed < n_splits
    class_order = np.random.RandomState(seed).permutation(num_classes).tolist()
    task_classes, cur = [], 0
    for sz in task_sizes:
        task_classes.append(class_order[cur: cur + sz]); cur += sz
    targets = np.asarray(train_targets)
    tr_idx, va_idx = [], []
    rng_split = np.random.RandomState(base_seed)
    for c in range(num_classes):
        perm = rng_split.permutation(np.where(targets == c)[0])
        folds = np.array_split(perm, n_splits)
        va_idx.extend(folds[seed])
        for k in range(n_splits):
            if k != seed: tr_idx.extend(folds[k])
    return task_classes, np.array(tr_idx), np.array(va_idx)

def split_dir(ds, phase, seed): return f"{WORK}/splits/{ds}/{phase}/seed{seed}"

def write_splits(ds, phase, seeds):
    sp = DS_SPEC[ds]
    tr_items, tr_y, te_items, te_y, root = load_source(ds)
    assert len(set(tr_y)) == sp["C"] and sum(sp["sizes"]) == sp["C"], f"{ds}: số lớp {len(set(tr_y))} ≠ {sp['C']}"
    rows, vals = [], {}
    for s in seeds:
        task_classes, tr_idx, va_idx = setup_seed(tr_y, s, sp["C"], sp["sizes"])
        order = sum(task_classes, [])
        pos = {c: i for i, c in enumerate(order)}                       # nhãn mới = vị trí trong thứ tự task
        if phase == "val":
            tr_sel = [(tr_items[i], tr_y[i]) for i in tr_idx]
            ev_sel = [(tr_items[i], tr_y[i]) for i in va_idx]
        else:
            tr_sel = list(zip(tr_items, tr_y))
            ev_sel = list(zip(te_items, te_y))
        d = split_dir(ds, phase, s); os.makedirs(d, exist_ok=True)
        for name, sel in (("train", tr_sel), ("test", ev_sel)):
            js = {"dataset": ds, "phase": phase, "seed": s, "kind": sp["kind"], "root": root, "class_order": order,
                  "task_sizes": sp["sizes"], "data": [x for x, _ in sel], "targets": [pos[y] for _, y in sel]}
            json.dump(js, open(f"{d}/{name}.json", "w"))
        vals[s] = set(va_idx.tolist())
        rows.append(dict(seed=s, train=len(tr_sel), eval=len(ev_sel), task1_5=task_classes[0][:5],
                         task_sizes=[len(t) for t in task_classes]))
    print(f"✅ {ds} ({phase}) | root: {root} | train gốc {len(tr_items)} / test {len(te_items)} | {sp['C']} lớp")
    for r in rows: print("  ", r)
    if phase == "val" and len(vals) > 1:
        ks = sorted(vals); print("   overlap Val giữa các seed:", [len(vals[a] & vals[b]) for a in ks for b in ks if a < b])
    return tr_items, te_items

def check_decode(ds, phase, seed, n=64):
    """Đọc thử n ảnh ngẫu nhiên của cả hai file json (train, test) để chắc chắn đường dẫn/ảnh dùng được."""
    bad = []
    for name in ("train", "test"):
        js = json.load(open(f"{split_dir(ds, phase, seed)}/{name}.json"))
        pick = random.Random(0).sample(range(len(js["data"])), min(n, len(js["data"])))
        if js["kind"] == "cifar": continue                                # pickle: đã đọc ở trên
        for i in pick:
            try:
                with Image.open(js["data"][i]) as im: im.convert("RGB").load()
            except Exception as e: bad.append((js["data"][i], repr(e)[:60]))
    assert not bad, f"{ds}: ảnh lỗi {bad[:3]}"

for ds in (DATASETS if RUN_TRAINING else []):
    write_splits(ds, PHASE, SEEDS)
    check_decode(ds, PHASE, SEEDS[0])
print("Dữ liệu sẵn sàng:", f"{WORK}/splits")


✅ CIFAR100 (val) | root: /kaggle/input/datasets/thohintrnhngc/cifar-100-python/cifar-100-python | train gốc 50000 / test 10000 | 100 lớp
   {'seed': 1, 'train': 40000, 'eval': 10000, 'task1_5': [80, 84, 33, 81, 93], 'task_sizes': [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]}
Dữ liệu sẵn sàng: /kaggle/working/apt_spherical/splits


## 3b. Patch repo và thư viện phân tích
Patch có assert: repo khác bản đã đọc thì dừng ngay. Repo đọc split của ô 3 qua biến môi trường `APT_SPLIT_DIR`; `rand_split=False` vì thứ tự task đã nằm trong nhãn.

In [4]:
"""Patch the official APT repo (in place): optional KAC head + dump of logits / frozen features after every task.
Env vars read at run time:  APT_HEAD=cosine|kac (default cosine = official),  APT_KAC_GRIDS (default 4),  APT_DUMP=<dir> (dump on),
APT_DUMP_APT=0|1|2 (also dump APT features; 2 = also re-embed old train images, reference only)."""
import os, re

def _sub(path, old, new, count=1):
    s = open(path).read()
    assert s.count(old) == count, f"{path}: anchor found {s.count(old)}x (expected {count}): {old[:60]!r}"
    open(path, "w").write(s.replace(old, new))

KAC_CLASS = '''
class KACHead(nn.Module):
    """KAC (CVPR'25): LayerNorm -> Gaussian RBF on evenly spaced grids in [-2, 2] -> linear without bias (zero init)."""
    def __init__(self, dim, n_cls, num_grids=4, grid_range=(-2.0, 2.0)):
        super().__init__(); gmin, gmax = grid_range
        self.layernorm = nn.LayerNorm(dim)
        self.register_buffer("grid", torch.linspace(gmin, gmax, num_grids))
        self.denominator = (gmax - gmin) / (num_grids - 1)
        self.spline_linear = nn.Linear(dim * num_grids, n_cls, bias=False); nn.init.zeros_(self.spline_linear.weight)
    def forward(self, f):
        x = self.layernorm(f)
        return self.spline_linear(torch.exp(-((x[..., None] - self.grid) / self.denominator) ** 2).flatten(1))

'''

DUMP_METHOD = '''
    @torch.no_grad()
    def _dump(self, i):
        """After task i: frozen (prompt-free) features of task-i TRAIN images, frozen features of task-i TEST images,
        and the model's logits (merged prompt, all seen classes) on the test images of tasks 0..i.
        APT_DUMP_APT=1|2 also dumps the APT features (CLS after the prompts = the vector the head sees):
          train_apt_{i}_{i}.pt  task-i train images, model after task i      (what a real continual run can store)
          test_apt_{i}_{j}.pt   task-j test images (j<=i), model after task i
        APT_DUMP_APT=2 adds train_apt_{i}_{j}.pt (j<i): OLD train images re-embedded by the current model. A real continual
        run does not have those images, so it is only a reference that separates feature drift from feature quality."""
        d = os.path.join(os.environ["APT_DUMP"], "seed%d" % self.seed); os.makedirs(d, exist_ok=True)
        mode = os.environ.get("APT_DUMP_APT", "0"); want_apt = mode != "0"
        model = self.learner.model; was_training = model.training; model.eval()
        valid = self.learner.valid_out_dim

        def run(ds, want_logits, want_apt, want_frozen=True):
            loader = DataLoader(ds, batch_size=128, shuffle=False, drop_last=False, num_workers=int(self.workers))
            F_, K_, A_, Y_ = [], [], [], []
            for x, y, _ in loader:
                x = x.cuda()
                if want_frozen: F_.append(model.feat(x)[:, 0, :].float().cpu())      # frozen ViT, no prompt
                if want_logits or want_apt:
                    o = model(x)
                    if want_logits: K_.append(o[:, :valid].float().cpu())
                    if want_apt:
                        assert hasattr(model, "feat_out"), "model has no feat_out (patch not applied?)"
                        A_.append(model.feat_out.float().cpu())                       # CLS after the prompts
                Y_.append(y)
            cat = lambda L: torch.cat(L) if L else None
            return cat(F_), cat(K_), cat(A_), torch.cat(Y_)

        tr_tf = self.train_dataset.transform
        self.train_dataset.transform = self.test_dataset.transform              # no augmentation for the prototypes
        for j in (range(i + 1) if mode == "2" else [i]):                        # ascending: task i is loaded last (as before)
            self.train_dataset.load_dataset(j, train=True)
            f, _, a, y = run(self.train_dataset, False, want_apt, want_frozen=(j == i))
            if j == i: torch.save({"f": f.half(), "y": y}, os.path.join(d, "train_frozen_%d.pt" % i))
            if want_apt: torch.save({"f": a.half(), "y": y}, os.path.join(d, "train_apt_%d_%d.pt" % (i, j)))
        self.train_dataset.transform = tr_tf
        for j in range(i + 1):
            self.test_dataset.load_dataset(j, train=True)
            f, k, a, y = run(self.test_dataset, True, want_apt)
            if j == i: torch.save({"f": f.half(), "y": y}, os.path.join(d, "test_frozen_%d.pt" % j))
            torch.save({"k": k.half(), "y": y}, os.path.join(d, "test_%d_%d.pt" % (i, j)))
            if want_apt: torch.save({"f": a.half(), "y": y}, os.path.join(d, "test_apt_%d_%d.pt" % (i, j)))
        model.train(was_training)

'''

def apply(repo):
    z = f"{repo}/models/zoo.py"; t = f"{repo}/trainer.py"; dl = f"{repo}/dataloaders/dataloader.py"; pl = f"{repo}/learners/prompt.py"
    if "APT_PATCHED" in open(z).read(): return "already patched"
    # lr riêng cho head (APT_HEAD_LR); không đặt thì giống repo: một lr chung
    _sub(pl, "        # create optimizers\n        self.optimizer = torch.optim.__dict__",
         "        if os.environ.get('APT_HEAD_LR'):\n"
         "            mod = self.model.module if len(self.config['gpuid']) > 1 else self.model\n"
         "            optimizer_arg['params'] = [{'params': list(mod.prompt.parameters()), 'lr': self.config['lr']},\n"
         "                                       {'params': list(mod.last.parameters()), 'lr': float(os.environ['APT_HEAD_LR'])}]\n"
         "        # create optimizers\n        self.optimizer = torch.optim.__dict__")
    _sub(pl, "from __future__ import print_function\n", "from __future__ import print_function\nimport os\n")
    # numpy>=2 (Colab): np.fromstring in binary mode was removed
    _sub(dl, "np.fromstring(image.tobytes(), dtype=np.uint8)", "np.frombuffer(image.tobytes(), dtype=np.uint8).copy()")
    _sub(z, "# note - ortho init has not been found", KAC_CLASS + "# APT_PATCHED\n# note - ortho init has not been found")
    _sub(z, "        self.last = nn.Linear(768, num_classes) \n",
         "        self.use_kac = os.environ.get('APT_HEAD', 'cosine') == 'kac'\n"
         "        self.last = KACHead(768, num_classes, int(os.environ.get('APT_KAC_GRIDS', 4))) if self.use_kac else nn.Linear(768, num_classes)\n")
    _sub(z, '            "last.weight",\n            "last.bias", \n            ] \n        else:',
         '            "last.weight",\n            "last.bias", \n            "last.layernorm.weight", "last.layernorm.bias", "last.spline_linear.weight",\n            ] \n        else:')
    _sub(z, '            "last.weight",\n            "last.bias", \n            ]\n\n        for name',
         '            "last.weight",\n            "last.bias", \n            "last.layernorm.weight", "last.layernorm.bias", "last.spline_linear.weight",\n            ]\n\n        for name')
    _sub(z, "        out = self.clf_norm(out)\n        wt_norm",
         "        self.feat_out = out.detach()\n        if self.use_kac:\n            return self.last(out)\n        out = self.clf_norm(out)\n        wt_norm")
    s = open(z).read()
    if not re.search(r"^import os\b", s, re.M): open(z, "w").write("import os\n" + s)
    _sub(t, "    def evaluate(self, avg_metrics):", DUMP_METHOD + "    def evaluate(self, avg_metrics):")
    _sub(t, "                acc_table.append(self.task_eval(j))\n            temp_table",
         "                acc_table.append(self.task_eval(j))\n            if os.environ.get('APT_DUMP'): self._dump(i)\n            temp_table")
    s = open(t).read()
    if not re.search(r"^import os\b", s, re.M): open(t, "w").write("import os\n" + s)
    return "patched"


"""Patch dataloaders/dataloader.py of the official APT repo: when env APT_SPLIT_DIR is set, CIFAR100 / CUB200 /
StanfordCars / ImageNet_R read their data from {APT_SPLIT_DIR}/train.json or test.json (written by the data cell).
Without APT_SPLIT_DIR the repo behaves exactly as before."""

SPLIT_PATCH = r'''

# ===== APT_SPLIT_PATCH: đọc split train/val/test của nhóm từ $APT_SPLIT_DIR/{train,test}.json =====
import json as _json
_CIFAR_CACHE = {}
def _apt_split_load(self):
    d = os.environ["APT_SPLIT_DIR"]
    js = _json.load(open(os.path.join(d, "train.json" if (self.train or self.validation) else "test.json")))
    self.targets = list(js["targets"])
    if js["kind"] == "cifar":
        out = []
        for item in js["data"]:
            split, i = item.split(":")
            if (js["root"], split) not in _CIFAR_CACHE:
                with open(os.path.join(js["root"], split), "rb") as f:
                    entry = pickle.load(f, encoding="latin1")
                _CIFAR_CACHE[(js["root"], split)] = entry["data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
            out.append(_CIFAR_CACHE[(js["root"], split)][int(i)])
        self.data = np.stack(out)
    else:
        self.data = list(js["data"])

def _apt_wrap_load(cls):
    orig = cls.load
    def load(self):
        if os.environ.get("APT_SPLIT_DIR"):
            return _apt_split_load(self)
        return orig(self)
    cls.load = load

for _cls in (iCIFAR10, iIMAGENET_R, iCUB200, iStanfordCars):   # iCIFAR100 kế thừa load của iCIFAR10
    _apt_wrap_load(_cls)
'''


def apply_split_patch(repo):
    p = f"{repo}/dataloaders/dataloader.py"
    s = open(p).read()
    if "APT_SPLIT_PATCH" in s:
        return "split patch: already applied"
    for name in ("class iCIFAR10(", "class iCIFAR100(", "class iIMAGENET_R(", "class iCUB200(", "class iStanfordCars("):
        assert name in s, f"dataloader.py: không thấy {name}"
    assert "def load(self)" in s.split("class iCIFAR100(")[0].split("class iCIFAR10(")[1], "iCIFAR10.load không như mong đợi"
    assert "def load" not in s.split("class iCIFAR100(")[1].split("class iIMAGENET_R(")[0], "iCIFAR100 có load riêng - cần patch thêm"
    open(p, "w").write(s + SPLIT_PATCH)
    return "split patch: applied"


print(apply(REPO)); print(apply_split_patch(REPO))
sys.path.insert(0, f"{REPO}")

patched
split patch: applied


In [5]:
"""Offline analysis of the dumps written by the patched APT repo (apt_patch.py).
Per seed dir:  train_frozen_{t}.pt {f,y}   test_frozen_{j}.pt {f,y}   test_{t}_{j}.pt {k,y}  (k = logits over the t+1 seen tasks).
Labels are already in task order (class c of task j lies in [lo_j, hi_j)), frozen/KAC rows of a test set are in the same order."""
import os, glob, math
import numpy as np
import torch
import torch.nn.functional as F

W_GRID = [0.25, 0.5, 0.75, 1.0]
KAPPA_GRID = [8, 16, 32, 64]
K_GRID = [1, 3, 5]
LAM_GRID = [0.5, 1.0, 2.0]
WIN_GRID = [0.0, 0.5]


def embed(f):
    return F.normalize(F.layer_norm(f.float(), (f.shape[1],)), dim=1)


def sph_kmeans(Z, k, g, iters=20):
    k = min(k, Z.shape[0])
    C = Z[torch.randperm(Z.shape[0], generator=g)[:k]].clone()
    for _ in range(iters):
        a = (Z @ C.T).argmax(1)
        newC = torch.zeros_like(C).index_add_(0, a, Z)
        empty = newC.norm(dim=1) < 1e-8
        if empty.any():
            newC[empty] = Z[torch.randint(0, Z.shape[0], (int(empty.sum()),), generator=g)]
        C = F.normalize(newC, dim=1)
    return C


def seeds_complete(run_dir):
    res = {}
    for d in sorted(glob.glob(os.path.join(run_dir, "seed*"))):
        T = len(glob.glob(os.path.join(d, "train_frozen_*.pt")))
        if T and all(os.path.exists(os.path.join(d, f"test_{t}_{j}.pt")) for t in range(T) for j in range(t + 1)) \
           and all(os.path.exists(os.path.join(d, f"test_frozen_{j}.pt")) for j in range(T)):
            res[int(os.path.basename(d)[4:])] = (d, T)
    return res


def block_lsm(x, bounds):
    out = torch.empty_like(x)
    for lo, hi in bounds:
        out[:, lo:hi] = F.log_softmax(x[:, lo:hi], 1)
    return out


def block_lse(ls, bounds):
    out = torch.empty_like(ls)
    for lo, hi in bounds:
        out[:, lo:hi] = torch.logsumexp(ls[:, lo:hi], 1, keepdim=True)
    return out


def _metrics(A, T):
    at = [A[:t + 1, t].mean() for t in range(T)]
    forg = np.mean([A[j, j] - A[j, T - 1] for j in range(T - 1)]) if T > 1 else 0.0   # = repo's calc_general_forgetting
    return float(np.mean(at)), float(at[-1]), float(forg)


def analyse_seed(d, T, seed):
    ld = lambda n: torch.load(os.path.join(d, n))
    tr = [ld(f"train_frozen_{t}.pt") for t in range(T)]
    te = [ld(f"test_frozen_{j}.pt") for j in range(T)]
    # ranh giới task lấy từ độ rộng logit (repo CUB có nhãn 1..200 nên 1 lớp rỗng, task chứa nó có 19 lớp nhưng vẫn chiếm 20 cột)
    Vs = [ld(f"test_{t}_{t}.pt")["k"].shape[1] for t in range(T)]
    bounds_all = [((Vs[t - 1] if t else 0), Vs[t]) for t in range(T)]
    C = Vs[-1]
    Ztr = torch.cat([embed(x["f"]) for x in tr]); ytr = torch.cat([x["y"] for x in tr]).long()
    for t, x in enumerate(tr):
        lo, hi = bounds_all[t]; assert x["y"].min() >= lo and x["y"].max() < hi, f"task {t}: nhãn train ngoài [{lo},{hi})"
    has = torch.zeros(C, dtype=torch.bool); has[ytr.unique()] = True
    print(f"    {int((~has).sum())} lớp không có ảnh train (bỏ qua):", (~has).nonzero().flatten().tolist())
    z0 = Ztr[:512]
    G = 1 - z0 @ z0.T
    inv_med = 1.0 / G[torch.triu(torch.ones(len(z0), len(z0), dtype=torch.bool), 1)].median().item()
    g = torch.Generator().manual_seed(seed)
    cen = {k: torch.zeros(C, k, Ztr.shape[1]) for k in K_GRID}
    for c in range(C):
        if not has[c]: continue
        Zc = Ztr[ytr == c]
        for k in K_GRID:
            Cc = sph_kmeans(Zc, k, g)
            cen[k][c] = Cc[torch.arange(k) % Cc.shape[0]]
    acc, tid = {}, np.zeros((T, T))
    def put(key, j, t, pred, y):
        acc.setdefault(key, np.zeros((T, T)))[j, t] = 100.0 * (pred == y).float().mean().item()
    for t in range(T):
        bt = bounds_all[:t + 1]; V = bt[-1][1]
        for j in range(t + 1):
            v = ld(f"test_{t}_{j}.pt"); kac, y = v["k"].float(), v["y"].long()
            assert kac.shape[1] == V and torch.equal(y, te[j]["y"].long()), f"seed {seed} step {t} task {j}: nhãn/đặc trưng không khớp"
            z = embed(te[j]["f"]); lo, hi = bounds_all[j]
            lk = F.log_softmax(kac, 1); lin_k = block_lsm(kac, bt)
            put(("KAC",), j, t, kac.argmax(1), y)
            blk = torch.full_like(kac, -1e9); blk[:, lo:hi] = kac[:, lo:hi]
            put(("oracle",), j, t, blk.argmax(1), y)
            for k in K_GRID:
                cos = torch.einsum("nd,ckd->nck", z, cen[k][:V])
                for kp in KAPPA_GRID:
                    s = torch.logsumexp(kp * inv_med * (cos - 1), dim=2)
                    s[:, ~has[:V]] = -1e4
                    ls = F.log_softmax(s, 1)
                    for w in W_GRID:
                        put(("fuse", w, kp, k), j, t, ((1 - w) * lk + w * ls).argmax(1), y)
                    lt = block_lse(ls, bt); lin_s = block_lsm(s, bt)
                    for lam in LAM_GRID:
                        for w_in in WIN_GRID:
                            put(("hier", lam, w_in, kp, k), j, t, (lam * lt + (1 - w_in) * lin_k + w_in * lin_s).argmax(1), y)
                    if k == 1 and kp == 16:
                        pt = s.argmax(1); tid[j, t] = 100.0 * ((pt >= lo) & (pt < hi)).float().mean().item()
    rows = []
    for key, A in acc.items():
        avg, last, forg = _metrics(A, T)
        r = dict(seed=seed, variant=key[0], w=np.nan, kappa=np.nan, k=np.nan, lam=np.nan, w_in=np.nan, avg=avg, last=last, forgetting=forg)
        if key[0] == "fuse": r.update(w=key[1], kappa=key[2], k=key[3])
        if key[0] == "hier": r.update(lam=key[1], w_in=key[2], kappa=key[3], k=key[4])
        rows.append(r)
    return rows, dict(seed=seed, task_id_acc_old_last=float(np.mean([tid[j, T - 1] for j in range(T - 1)])) if T > 1 else float("nan"))


def analyse(run_dir):
    rows, diag = [], []
    for seed, (d, T) in sorted(seeds_complete(run_dir).items()):
        print(f"  seed {seed}: {T} task", flush=True)
        r, dg = analyse_seed(d, T, seed); rows += r; diag.append(dg)
    return rows, diag


In [6]:
"""Spherical branch on three feature sources, same KAC logits, same grids (needs fusion_online.py code in the same namespace).
  frozen      : ViT without prompt. Centres = train images of each task (once). Test features = frozen features.
  apt_stale   : APT features (CLS after the prompts). Centre of a class = APT features of its train images taken right after
                its own task and never refreshed; test features = APT features of the CURRENT model (what a real continual run has).
  apt_refresh : at every step, centres rebuilt from the current model's features of ALL seen train images.
                A real continual run does not keep old train images -> reference only (separates drift from feature quality).
Dumps: train_frozen_{t}.pt, train_apt_{t}_{j}.pt, test_frozen_{j}.pt, test_apt_{t}_{j}.pt, test_{t}_{j}.pt (see apt_patch.py)."""

SOURCES = ["frozen", "apt_stale", "apt_refresh"]


def _build_cen(Z, y, C, g):
    has = torch.zeros(C, dtype=torch.bool); has[y.unique()] = True
    cen = {k: torch.zeros(C, k, Z.shape[1]) for k in K_GRID}
    for c in has.nonzero().flatten().tolist():
        Zc = Z[y == c]
        for k in K_GRID:
            Cc = sph_kmeans(Zc, k, g); cen[k][c] = Cc[torch.arange(k) % Cc.shape[0]]
    return cen, has


def _inv_med(Z):
    z0 = Z[:512]; G = 1 - z0 @ z0.T
    return 1.0 / G[torch.triu(torch.ones(len(z0), len(z0), dtype=torch.bool), 1)].median().item()


def sources_available(d, T):
    ex = lambda n: os.path.exists(os.path.join(d, n))
    out = ["frozen"]
    if all(ex(f"train_apt_{t}_{t}.pt") for t in range(T)) and all(ex(f"test_apt_{t}_{j}.pt") for t in range(T) for j in range(t + 1)):
        out.append("apt_stale")
        if all(ex(f"train_apt_{t}_{j}.pt") for t in range(T) for j in range(t + 1)): out.append("apt_refresh")
    return out


def analyse_seed_src(d, T, seed, source):
    ld = lambda n: torch.load(os.path.join(d, n))
    Vs = [ld(f"test_{t}_{t}.pt")["k"].shape[1] for t in range(T)]
    bounds_all = [((Vs[t - 1] if t else 0), Vs[t]) for t in range(T)]
    C = Vs[-1]
    te_y = [ld(f"test_frozen_{j}.pt")["y"].long() for j in range(T)]
    g = torch.Generator().manual_seed(seed)
    if source == "frozen":
        tr = [ld(f"train_frozen_{t}.pt") for t in range(T)]
        zte_cache = [embed(ld(f"test_frozen_{j}.pt")["f"]) for j in range(T)]
        zte = lambda t, j: zte_cache[j]
    else:
        tr = [ld(f"train_apt_{t}_{t}.pt") for t in range(T)]
        zte = lambda t, j: embed(ld(f"test_apt_{t}_{j}.pt")["f"])
    for t, x in enumerate(tr):
        lo, hi = bounds_all[t]; assert x["y"].min() >= lo and x["y"].max() < hi, f"task {t}: nhãn train ngoài [{lo},{hi})"
    if source != "apt_refresh":                                   # centres built once, from each task's own train images
        Ztr = torch.cat([embed(x["f"]) for x in tr]); ytr = torch.cat([x["y"] for x in tr]).long()
        cen_s, has_s = _build_cen(Ztr, ytr, C, g); inv_s = _inv_med(Ztr)
        if source == "apt_stale":
            fz = [ld(f"train_frozen_{t}.pt")["y"] for t in range(T)]
            assert all(torch.equal(a["y"], b) for a, b in zip(tr, fz)), "train_apt và train_frozen lệch thứ tự"
    acc, tid = {}, np.zeros((T, T)); drift = np.full((T, T), np.nan)
    def put(key, j, t, pred, y):
        acc.setdefault(key, np.zeros((T, T)))[j, t] = 100.0 * (pred == y).float().mean().item()
    for t in range(T):
        bt = bounds_all[:t + 1]; V = bt[-1][1]
        if source == "apt_refresh":                               # re-embed everything seen so far with the current model
            parts = [ld(f"train_apt_{t}_{j}.pt") for j in range(t + 1)]
            Zt = torch.cat([embed(x["f"]) for x in parts]); yt = torch.cat([x["y"] for x in parts]).long()
            cen, has = _build_cen(Zt, yt, V, g); inv_med = _inv_med(embed(parts[0]["f"]))
        else:
            cen, has, inv_med = cen_s, has_s, inv_s
        for j in range(t + 1):
            v = ld(f"test_{t}_{j}.pt"); kac, y = v["k"].float(), v["y"].long()
            assert kac.shape[1] == V and torch.equal(y, te_y[j]), f"seed {seed} step {t} task {j}: nhãn/đặc trưng không khớp"
            z = zte(t, j); lo, hi = bounds_all[j]
            if source == "apt_stale" and j < t:                   # how far did the same test images move since task j?
                drift[j, t] = (z * zte(j, j)).sum(1).mean().item()
            lk = F.log_softmax(kac, 1); lin_k = block_lsm(kac, bt)
            put(("KAC",), j, t, kac.argmax(1), y)
            blk = torch.full_like(kac, -1e9); blk[:, lo:hi] = kac[:, lo:hi]
            put(("oracle",), j, t, blk.argmax(1), y)
            for k in K_GRID:
                cos = torch.einsum("nd,ckd->nck", z, cen[k][:V])
                for kp in KAPPA_GRID:
                    s = torch.logsumexp(kp * inv_med * (cos - 1), dim=2)
                    s[:, ~has[:V]] = -1e4
                    ls = F.log_softmax(s, 1)
                    for w in W_GRID:
                        put(("fuse", w, kp, k), j, t, ((1 - w) * lk + w * ls).argmax(1), y)
                    lt = block_lse(ls, bt); lin_s = block_lsm(s, bt)
                    for lam in LAM_GRID:
                        for w_in in WIN_GRID:
                            put(("hier", lam, w_in, kp, k), j, t, (lam * lt + (1 - w_in) * lin_k + w_in * lin_s).argmax(1), y)
                    if k == 1 and kp == 16:
                        pt = s.argmax(1); tid[j, t] = 100.0 * ((pt >= lo) & (pt < hi)).float().mean().item()
    rows = []
    for key, A in acc.items():
        avg, last, forg = _metrics(A, T)
        r = dict(seed=seed, source=source, variant=key[0], w=np.nan, kappa=np.nan, k=np.nan, lam=np.nan, w_in=np.nan, avg=avg, last=last, forgetting=forg)
        if key[0] == "fuse": r.update(w=key[1], kappa=key[2], k=key[3])
        if key[0] == "hier": r.update(lam=key[1], w_in=key[2], kappa=key[3], k=key[4])
        rows.append(r)
    dg = dict(seed=seed, source=source,
              task_id_acc_old_last=float(np.mean([tid[j, T - 1] for j in range(T - 1)])) if T > 1 else float("nan"),
              cos_same_image_old_task_last=float(np.nanmean([drift[j, T - 1] for j in range(T - 1)])) if (source == "apt_stale" and T > 1) else float("nan"))
    return rows, dg


def analyse_all(run_dir, sources=SOURCES):
    rows, diag = [], []
    for seed, (d, T) in sorted(seeds_complete(run_dir).items()):
        have = sources_available(d, T)
        for s in sources:
            if s not in have:
                print(f"  seed {seed}: thiếu dump cho nguồn {s} (cần APT_DUMP_APT={'2' if s == 'apt_refresh' else '1'}) - bỏ qua"); continue
            print(f"  seed {seed}: {T} task, nguồn {s}", flush=True)
            r, dg = analyse_seed_src(d, T, seed, s); rows += r; diag.append(dg)
    return rows, diag


## 4. Hàm phụ và chạy thử nhanh (~10 phút)
Ô hàm phụ luôn phải chạy. Ô chạy thử: 2 task × 2 epoch trên `DATASETS[0]`, seed `SEEDS[0]` - kiểm tra train + dump + phân tích chạy thông (có thể bỏ qua).

In [7]:
# ==============================================================================
# 4a. HÀM PHỤ (config cho repo, lệnh chạy, biến môi trường) - luôn chạy ô này
# ==============================================================================
import yaml
def make_cfg(ds, max_task=-1, tag=None):
    """Config của repo + rand_split=False (thứ tự task đã nằm trong nhãn) + kích thước task của nhóm."""
    cfg = yaml.safe_load(open(f"{REPO}/{HP[ds]['config']}"))
    sizes = DS_SPEC[ds]["sizes"]
    cfg.update(rand_split=False, first_split_size=sizes[0], other_split_size=sizes[1], max_task=max_task, validation=False)
    path = f"configs/_{tag or ds}.yaml"; yaml.safe_dump(cfg, open(f"{REPO}/{path}", "w")); return path

def cmd(ds, seed, out, config, epochs=None):
    h = HP[ds]
    return [sys.executable, "-u", "run.py", "--config", config, "--gpuid", "0", "--repeat", "1", "--overwrite", "0",
            "--learner_type", "prompt", "--learner_name", "APT_Learner", "--prompt_param", str(PROMPT_PARAM),
            "--lr", str(h["lr"]), "--seed", str(seed), "--ema_coeff", str(h["ema"]),
            "--schedule", str(epochs or h["epochs"]), "--batch_size", str(BATCH), "--log_dir", out]

def run_env(ds, seed, dumps, head="kac"):
    return dict(os.environ, APT_HEAD=head, APT_KAC_GRIDS=str(KAC_GRIDS), APT_DUMP=dumps, APT_DUMP_APT=APT_DUMP_APT,
                APT_SPLIT_DIR=split_dir(ds, PHASE, seed), **({"APT_HEAD_LR": str(HEAD_LR)} if (HEAD_LR and head == "kac") else {}))



In [8]:
# ==============================================================================
# 4. CHẠY THỬ NHANH (~10 phút): DATASETS[0], seed SEEDS[0], 2 task × 2 epoch
# ==============================================================================
if RUN_TRAINING and RUN_SMOKE:
    SMOKE_DS = DATASETS[0]
    shutil.rmtree("/tmp/smoke", ignore_errors=True)
    r = subprocess.run(cmd(SMOKE_DS, SEEDS[0], "/tmp/smoke/out", make_cfg(SMOKE_DS, max_task=2, tag="smoke"), epochs=2),
                       cwd=REPO, env=run_env(SMOKE_DS, SEEDS[0], "/tmp/smoke/dumps"), capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-1500:] if r.returncode else "")
    assert r.returncode == 0, "chạy thử lỗi - gửi mình đoạn lỗi ở trên"
    assert "Shuffling" not in r.stdout, "repo vẫn tự xáo lớp (rand_split) - thứ tự task sẽ sai"
    rows, diag = analyse_all("/tmp/smoke/dumps")
    import pandas as pd
    print(pd.DataFrame(rows).groupby(["source", "variant"])[["avg", "last"]].max().round(1)); print(pd.DataFrame(diag))
    print("Chạy thử OK (số liệu 2 epoch chưa có ý nghĩa)")


## 5. Chạy thật

In [9]:
# ==============================================================================
# 5. CHẠY THẬT: mỗi (dataset, seed) ghi vào {DRIVE_DIR}/runs/kac/<ds>/<PHASE>/ ; seed đã xong được bỏ qua
# ==============================================================================
import time
def done(out): return os.path.exists(f"{out}/results-acc/global.yaml")
if RUN_TRAINING:
    for ds in DATASETS:
        config = make_cfg(ds)
        for seed in SEEDS:
            base = f"{DRIVE_DIR}/runs/kac/{ds}/{PHASE}"; out, dumps = f"{base}/seed{seed}", f"{base}/dumps"
            if done(out): print(ds, PHASE, seed, "đã xong, bỏ qua"); continue
            shutil.rmtree(out, ignore_errors=True); shutil.rmtree(f"{dumps}/seed{seed}", ignore_errors=True); os.makedirs(out)
            t0 = time.time(); print(f"== {ds} | {PHASE} | seed {seed} ==", flush=True)
            r = subprocess.run(cmd(ds, seed, out, config), cwd=REPO, env=run_env(ds, seed, dumps))
            shutil.rmtree(f"{out}/models", ignore_errors=True)
            print(f"   xong sau {(time.time()-t0)/60:.1f} phút, mã thoát {r.returncode}", flush=True)
            assert r.returncode == 0, f"lỗi - xem {out}/output.log"


== CIFAR100 | val | seed 1 ==


/tmp/apt_repo/Additive-Prompt-Tuning-main/run.py:192: SyntaxWarning: invalid escape sequence '\p'
  print(round(avg_metrics[mkey]['global'][-1,:r+1].mean(),2), '\pm', round(avg_metrics[mkey]['global'][-1,:r+1].std(),2))
/tmp/apt_repo/Additive-Prompt-Tuning-main/run.py:210: SyntaxWarning: invalid escape sequence '\p'
  print("forgetting ,",round(mean_forgetting,2), "\pm",round(std_forgetting,2))


Namespace(dataroot='data', dataset='CIFAR100', first_split_size=10, max_task=-1, model_name='vit_pt_imnet', model_type='zoo', momentum=0.9, optimizer='Adam', other_split_size=10, rand_split=False, schedule_type='cosine', train_aug=True, validation=False, weight_decay=0, workers=4, gpuid=[0], log_dir='/kaggle/working/apt_spherical/runs/kac/CIFAR100/val/seed1', learner_type='prompt', learner_name='APT_Learner', debug_mode=0, repeat=1, overwrite=0, oracle_flag=False, upper_bound_flag=False, memory=0, temp=2.0, DW=False, prompt_param=['0.01'], seed=1, batch_size=64, lr=0.004, ema_coeff=0.7, schedule=30, config='configs/_CIFAR100.yaml')
************************************
* STARTING TRIAL 1
************************************
*****************************************
====================== 1 =======================
Incremental class: Old valid output dimension: 0
Incremental class: New Valid output dimension: 10
Overwriting ...
Optimizer is reset!
*****************************************

/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:3904: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:147: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:227: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:184: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.13/dist-packages/numpy/_core/_methods.py:219: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


   xong sau 664.9 phút, mã thoát 0


## 6. Phân tích và bảng kết quả
`PHASE="val"`: chọn cấu hình theo Last trung bình các seed trên Val, lưu `best_configs_val.json`. `PHASE="test"`: báo cáo trên Test với đúng cấu hình đó.

In [10]:
import os, sys, glob, shutil, json
# ==============================================================================
# 6. PHÂN TÍCH: PHASE="val" -> chọn cấu hình (theo Last trung bình 3 seed) và lưu best_configs_val.json
#               PHASE="test" -> dùng đúng cấu hình đã chọn ở "val", báo cáo trên Test
# Gom dump của mọi seed: phiên hiện tại + output của phiên trước (Add Input -> notebook output).
# ==============================================================================
import json, pandas as pd
def gather_dumps(ds, phase):
    dst = f"/kaggle/working/_merged/{ds}/{phase}"; shutil.rmtree(dst, ignore_errors=True); os.makedirs(dst)
    for pat in (f"{DRIVE_DIR}/runs/kac/{ds}/{phase}/dumps", f"/kaggle/input/**/runs/kac/{ds}/{phase}/dumps"):
        for parent in sorted(glob.glob(pat, recursive=True)):
            for s, (d, T) in seeds_complete(parent).items():
                if s in SEEDS and not os.path.exists(f"{dst}/seed{s}"): os.symlink(os.path.abspath(d), f"{dst}/seed{s}")
    return dst

KEYS = ["variant", "w", "kappa", "k", "lam", "w_in"]
VARIANTS = {"proto": ("fuse", {"w": 1.0}), "fuse": ("fuse", {}), "hier": ("hier", {})}
ROWS, DIAG, AG = {}, {}, {}
for ds in DATASETS:
    dd = gather_dumps(ds, PHASE)
    print(f"===== {ds} | {PHASE} | seed có đủ dump: {sorted(os.listdir(dd))}", flush=True)
    if not os.listdir(dd): continue
    ROWS[ds], DIAG[ds] = analyse_all(dd)
    d = pd.DataFrame(ROWS[ds])
    AG[ds] = d.groupby(["source"] + KEYS, dropna=False).agg(seeds=("seed", "nunique"), avg=("avg", "mean"), avg_sd=("avg", "std"),
        last=("last", "mean"), last_sd=("last", "std"), forg=("forgetting", "mean"), forg_sd=("forgetting", "std")).reset_index()
    AG[ds].assign(dataset=ds, phase=PHASE).to_csv(f"{DRIVE_DIR}/fusion_apt_summary_{ds}_{PHASE}.csv", index=False)

clean = lambda r: {k: (None if pd.isna(r[k]) else (float(r[k]) if k != "variant" else r[k])) for k in KEYS}
BEST_PATH = f"{DRIVE_DIR}/best_configs_val.json"
def load_best():
    """Gộp best_configs_val.json của mọi phiên PHASE="val" (Add Input output các phiên trước) + phiên hiện tại."""
    best = {}
    for p in sorted(glob.glob("/kaggle/input/**/best_configs_val.json", recursive=True)) + [BEST_PATH]:
        if os.path.exists(p): best.update(json.load(open(p))); print("  đọc cấu hình:", p)
    return best
BEST = load_best()
if PHASE == "val":
    for ds, ag in AG.items():
        BEST[ds] = {}
        for src in [s for s in SOURCES if s in set(ag.source)]:
            BEST[ds][src] = {}
            for v, (vv, cond) in VARIANTS.items():
                m = ag[(ag.source == src) & (ag.variant == vv)]
                for a, b in cond.items(): m = m[m[a] == b]
                BEST[ds][src][v] = clean(m.sort_values("last", ascending=False).iloc[0])
    json.dump(BEST, open(BEST_PATH, "w"), indent=1); print("Đã lưu cấu hình chọn trên Val:", BEST_PATH, "| dataset:", sorted(BEST))
else:
    miss = [ds for ds in AG if ds not in BEST]
    assert not miss, f"PHASE='test' cần cấu hình chọn trên Val cho {miss}: Add Input output của notebook chạy PHASE='val' (có best_configs_val.json)"

def pick(ag, src, cfg):
    m = ag[ag.source == src]
    for k in KEYS: m = m[m[k].isna()] if cfg[k] is None else m[m[k] == cfg[k]]
    return m.iloc[0]
REPORT = []
for ds, ag in AG.items():
    if ds not in BEST: print(ds, ": chưa có cấu hình chọn trên Val -> bỏ qua"); continue
    base = ag[ag.source == "frozen"]
    for name in ("KAC", "oracle"):
        r = base[base.variant == name].iloc[0]; REPORT.append(dict(dataset=ds, source="-", method=name, **{c: r[c] for c in ("seeds", "avg", "avg_sd", "last", "last_sd", "forg", "forg_sd")}, config=""))
    for src, vs in BEST[ds].items():
        if src not in set(ag.source): continue
        for v, cfg in vs.items():
            r = pick(ag, src, cfg)
            REPORT.append(dict(dataset=ds, source=src, method=v, **{c: r[c] for c in ("seeds", "avg", "avg_sd", "last", "last_sd", "forg", "forg_sd")},
                               config=", ".join(f"{k}={cfg[k]:g}" for k in KEYS[1:] if cfg[k] is not None)))
REP = pd.DataFrame(REPORT)
if len(REP):
    REP.to_csv(f"{DRIVE_DIR}/report_{PHASE}.csv", index=False)
    fmt = lambda r: f"{r.avg:6.2f}±{r.avg_sd:4.2f} | {r['last']:6.2f}±{r.last_sd:4.2f} | {r.forg:6.2f}±{r.forg_sd:4.2f}"
    for ds in REP.dataset.unique():
        print(f"\n===== {ds} | {PHASE} | Avg | Last | Forgetting =====")
        for _, r in REP[REP.dataset == ds].iterrows(): print(f"  {r.source:12} {r.method:7}", fmt(r.fillna(0)), f"  [{r.config}]" if r.config else "")
        dg = pd.DataFrame(DIAG[ds]).groupby("source").mean(numeric_only=True).drop(columns="seed")
        print(dg.rename(columns={"task_id_acc_old_last": "chọn đúng task (%)", "cos_same_image_old_task_last": "cos cùng ảnh (drift)"}).round(3))


===== CIFAR100 | val | seed có đủ dump: ['seed1']
  seed 1: 10 task, nguồn frozen
  seed 1: 10 task, nguồn apt_stale
  seed 1: 10 task, nguồn apt_refresh
Đã lưu cấu hình chọn trên Val: /kaggle/working/apt_spherical/best_configs_val.json | dataset: ['CIFAR100']

===== CIFAR100 | val | Avg | Last | Forgetting =====
  -            KAC      92.51±0.00 |  87.53±0.00 |   4.80±0.00 
  -            oracle   98.53±0.00 |  98.49±0.00 |   0.03±0.00 
  frozen       proto    86.08±0.00 |  80.20±0.00 |   5.81±0.00   [w=1, kappa=16, k=5]
  frozen       fuse     93.10±0.00 |  88.75±0.00 |   4.44±0.00   [w=0.25, kappa=32, k=5]
  frozen       hier     91.15±0.00 |  85.07±0.00 |   5.61±0.00   [kappa=8, k=5, lam=0.5, w_in=0]
  apt_stale    proto    91.91±0.00 |  84.89±0.00 |  10.89±0.00   [w=1, kappa=8, k=1]
  apt_stale    fuse     93.37±0.00 |  88.39±0.00 |   6.37±0.00   [w=0.25, kappa=16, k=3]
  apt_stale    hier     92.56±0.00 |  85.53±0.00 |  10.63±0.00   [kappa=8, k=1, lam=0.5, w_in=0]
  apt_refresh 

## 7. Gói kết quả nhỏ để gửi lại

In [11]:
# ==============================================================================
# 7. GÓI KẾT QUẢ NHỎ (gửi lại): bảng tổng hợp, cấu hình chọn trên Val, log của repo
# ==============================================================================
import zipfile
zp = f"{DRIVE_DIR}/apt_spherical_results_{PHASE}.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob(f"{DRIVE_DIR}/*.csv") + glob.glob(f"{DRIVE_DIR}/*.json"): z.write(p, os.path.basename(p))
    for p in glob.glob(f"{DRIVE_DIR}/runs/*/*/*/seed*/output.log"): z.write(p, os.path.relpath(p, DRIVE_DIR))
print("Đã tạo", zp)


Đã tạo /kaggle/working/apt_spherical/apt_spherical_results_val.zip
